# US Accidents: Analysis and Geographic Clustering

This notebook studies reported US accidents from 2016 through March 2023 using Polars, Parquet, PySpark, Plotly, and Dash. It uses a reproducible (fixed random seed) 20% severity-stratified sample so the workflow remains practical on a local computer.

The analysis asks how reported accidents vary by severity, location, time, weather, and selected road features. It also uses K-Means to form broad geographic regions from accident coordinates.

> Important: reported accident counts describe frequency in this dataset, not accident risk. Risk estimates would require additional information such as traffic volume or distance driven, etc...






### Dataset Source:
**Dataset:** US Accidents (2016–2023)  
**Author:** Sobhan Moosavi  
**Source:** Kaggle
(https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents/data?utm_source.com). 
**DOI:** 10.34740/KAGGLE/DS/199387  
**License:** CC BY-NC-SA 4.0


# 1. Setup

The setup first handles the Java and Python environment required by PySpark.  
After that, the project libraries and main configuration settings are loaded.

### Environment compatibility setup


In [1]:
# Environment compatibility setup for PySpark.
# PySpark in this project is expected to run with Java 17.
# This block searches for an existing Java 17 installation in common Conda and JAVA_HOME locations,
# checks the Java version, configures the notebook to use it,
# and makes PySpark use the same Python interpreter as the current notebook kernel.

import os
import re
import subprocess
import sys
from pathlib import Path


java_binary = 'java.exe' if os.name == 'nt' else 'java'


def inspect_java(java_home: Path):
    """Return (major_version, full_version_text) for a Java home, if valid."""

    # Build the path to the Java executable inside this candidate Java installation.
    java_executable = java_home / 'bin' / java_binary

    # If Java does not exist in this location, ignore it.
    if not java_executable.exists():
        return None, None

    # Run "java -version" to read the installed Java version.
    result = subprocess.run(
        [str(java_executable), '-version'],
        capture_output=True,
        text=True,
        check=True,
    )

    # Extract the major version number, for example 17 from version 17.0.x.
    version_text = (result.stderr or result.stdout).strip()
    match = re.search(r'version "([0-9]+)', version_text)
    major_version = int(match.group(1)) if match else None

    return major_version, version_text


# Build a list of common places where Java may be installed.
candidate_java_homes = [
    Path(sys.prefix) / 'Library',  # Common Conda layout on Windows
    Path(sys.prefix),              # Common Conda/Linux layout
]


# Also check paths already defined by Conda or JAVA_HOME.
for environment_name in ('CONDA_PREFIX', 'JAVA_HOME'):
    environment_path = os.environ.get(environment_name)

    if environment_path:
        candidate_java_homes.extend([
            Path(environment_path) / 'Library',
            Path(environment_path),
        ])


# Search the candidate locations and stop at the first valid Java 17 installation.
java_home = None
java_version_text = None

for candidate in dict.fromkeys(candidate_java_homes):
    major_version, version_text = inspect_java(candidate)

    if major_version == 17:
        java_home = candidate
        java_version_text = version_text
        break


# If Java 17 was not found, stop the notebook with a clear error message.
if java_home is None:
    raise RuntimeError(
        'Java 17 was not found in the active Python/Conda environment. '
        'Install or select an environment containing Java 17, restart the kernel, '
        'and run the notebook again.'
    )


# Tell PySpark to use the Java 17 installation that was found.
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home / 'bin') + os.pathsep + os.environ.get('PATH', '')


# Make Spark workers use the same Python interpreter as this notebook kernel.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


# Show the environment selected for PySpark.
print(f'JAVA_HOME: {java_home}')
print(f'Java: {java_version_text.splitlines()[0]}')

JAVA_HOME: c:\Users\LENOVO\anaconda3\envs\GPU_DEEP_LEARNING\Library
Java: openjdk version "17.0.18" 2026-01-20 LTS


### Project imports

In [2]:
import polars as pl
import plotly.express as px
from dash import Dash, Input, Output, dcc, html

# Import PySpark only after the Java environment has been configured.
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count

### Project configuration

In [3]:
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(30)


SELECTED_COLUMNS = [
    # Identification
    'ID',

    # Accident impact
    'Severity',
    'Distance(mi)',

    # Time
    'Start_Time',
    'End_Time',
    'Sunrise_Sunset',

    # Location
    'Start_Lat',
    'Start_Lng',
    'City',
    'State',

    # Weather
    'Temperature(F)',
    'Humidity(%)',
    'Visibility(mi)',
    'Wind_Speed(mph)',
    'Weather_Condition',

    # Road features
    'Crossing',
    'Junction',
    'Traffic_Signal',
]

# Only these fields are required for the notebook's main analyses.
# Missing weather measurements are handled later in weather-specific summaries.
CORE_REQUIRED = [
    'ID', 'Severity', 'Start_Time', 'End_Time',
    'Start_Lat', 'Start_Lng', 'City', 'State',
]


WEATHER_COLUMNS = [
    'Temperature(F)',
    'Humidity(%)',
    'Visibility(mi)',
    'Wind_Speed(mph)',
]


ROAD_FEATURES = [
    'Crossing',
    'Junction',
    'Traffic_Signal',
]


TIME_PERIOD_ORDER = [
    'Morning',
    'Afternoon',
    'Evening',
    'Night',
]


CONFIG = {
    'input_path': Path('US_Accidents_March23.csv'),
    'artifact_dir': Path('outputs'),
    'processed_path': Path('outputs/accidents_processed.parquet'),
    'sample_fraction': 0.20,
    'seed': 42,
    'k_values': range(3, 13),
    'map_sample_size': 10_000,
    'cluster_map_sample_size': 20_000,
    'export_figures': False,
}


print('Setup complete.')

Setup complete.


## 2. Data Loading

Only the columns needed for the analysis are loaded from the CSV to reduce memory usage.

In [4]:
if not CONFIG['input_path'].exists():
    raise FileNotFoundError(
        f"Dataset not found: {CONFIG['input_path']}. "
        'Place the CSV beside this notebook or update CONFIG.'
    )

source_accidents = pl.read_csv(
    CONFIG['input_path'],
    columns=SELECTED_COLUMNS,
)

print(f'Selected source rows: {source_accidents.height:,}')
print(f'Selected columns: {source_accidents.width}')
display(source_accidents.head())
display(pl.DataFrame({
    'Column': source_accidents.columns,
    'Data_Type': [str(dtype) for dtype in source_accidents.dtypes],
}))

Selected source rows: 7,728,394
Selected columns: 18


ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,State,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Weather_Condition,Crossing,Junction,Traffic_Signal,Sunrise_Sunset
str,i64,str,str,f64,f64,f64,str,str,f64,f64,f64,f64,str,bool,bool,bool,str
"""A-1""",3,"""2016-02-08 05:46:00""","""2016-02-08 11:00:00""",39.865147,-84.058723,0.01,"""Dayton""","""OH""",36.9,91.0,10.0,null,"""Light Rain""",false,false,false,"""Night"""
"""A-2""",2,"""2016-02-08 06:07:59""","""2016-02-08 06:37:59""",39.928059,-82.831184,0.01,"""Reynoldsburg""","""OH""",37.9,100.0,10.0,null,"""Light Rain""",false,false,false,"""Night"""
"""A-3""",2,"""2016-02-08 06:49:27""","""2016-02-08 07:19:27""",39.063148,-84.032608,0.01,"""Williamsburg""","""OH""",36.0,100.0,10.0,3.5,"""Overcast""",false,false,true,"""Night"""
"""A-4""",3,"""2016-02-08 07:23:34""","""2016-02-08 07:53:34""",39.747753,-84.205582,0.01,"""Dayton""","""OH""",35.1,96.0,9.0,4.6,"""Mostly Cloudy""",false,false,false,"""Night"""
"""A-5""",2,"""2016-02-08 07:39:07""","""2016-02-08 08:09:07""",39.627781,-84.188354,0.01,"""Dayton""","""OH""",36.0,89.0,6.0,3.5,"""Mostly Cloudy""",false,false,true,"""Day"""


Column,Data_Type
str,str
"""ID""","""String"""
"""Severity""","""Int64"""
"""Start_Time""","""String"""
"""End_Time""","""String"""
"""Start_Lat""","""Float64"""
"""Start_Lng""","""Float64"""
"""Distance(mi)""","""Float64"""
"""City""","""String"""
"""State""","""String"""


## 3. Data Quality Check

Missing values and the severity distribution are checked before sampling and cleaning so we can compare them with the data later.

In [5]:
missing_values = (
    source_accidents.null_count()
    .transpose(
        include_header=True,
        header_name='Column',
        column_names=['Missing_Rows'],
    )
    .with_columns(
        (pl.col('Missing_Rows') / source_accidents.height * 100)
        .round(2)
        .alias('Missing_%')
    )
    .sort('Missing_%', descending=True)
)

severity_before_sampling = (
    source_accidents.group_by('Severity')
    .len()
    .rename({'len': 'Source_Rows'})
    .sort('Severity')
)

print('Missing values in the selected source columns')
display(missing_values)
print('Source severity distribution')
display(severity_before_sampling)

Missing values in the selected source columns


Column,Missing_Rows,Missing_%
str,u32,f64
"""Wind_Speed(mph)""",571233,7.39
"""Visibility(mi)""",177098,2.29
"""Humidity(%)""",174144,2.25
"""Weather_Condition""",173459,2.24
"""Temperature(F)""",163853,2.12
"""Sunrise_Sunset""",23246,0.3
"""ID""",0,0.0
"""Severity""",0,0.0
"""Start_Time""",0,0.0


Source severity distribution


Severity,Source_Rows
i64,u32
1,67366
2,6156981
3,1299337
4,204710


## 4. Severity-Stratified Sampling

20% of the data is sampled from each severity level. Stratification helps the sample retain the original severity mix, while the fixed seed gives the same sample each time.

In [6]:
sampled_accidents = (
    source_accidents.group_by('Severity')
    .map_groups(
        lambda severity_group: severity_group.sample(
            fraction=CONFIG['sample_fraction'],
            seed=CONFIG['seed'],
        )
    )
)

severity_after_sampling = (
    sampled_accidents.group_by('Severity')
    .len()
    .rename({'len': 'Sampled_Rows'})
    .sort('Severity')
)

sampling_audit = (
    severity_before_sampling
    .join(severity_after_sampling, on='Severity')
    .with_columns(
        (pl.col('Sampled_Rows') / pl.col('Source_Rows') * 100)
        .round(2)
        .alias('Sampled_%')
    )
)

print(f'Sampled rows: {sampled_accidents.height:,}')
display(sampling_audit)

# The full selected dataset is no longer needed after the sample is created.
del source_accidents

Sampled rows: 1,545,678


Severity,Source_Rows,Sampled_Rows,Sampled_%
i64,u32,u32,f64
1,67366,13473,20.0
2,6156981,1231396,20.0
3,1299337,259867,20.0
4,204710,40942,20.0


## 5. Preprocessing

Timestamps are parsed first. Rows are removed only when a field required by the main workflow is missing. Accident duration is then calculated in minutes, and records with negative durations or durations above 24 hours are excluded as implausible for this analysis.

Weather columns are deliberately not part of the core missing-value filter.

In [7]:
parsed_accidents = sampled_accidents.with_columns([
    pl.col('Start_Time').str.to_datetime(strict=False),
    pl.col('End_Time').str.to_datetime(strict=False),
])

core_clean_accidents = parsed_accidents.drop_nulls(CORE_REQUIRED)

duration_clean_accidents = (
    core_clean_accidents
    .with_columns(
        (
            (pl.col('End_Time') - pl.col('Start_Time')).dt.total_seconds() / 60
        ).alias('Duration_Minutes')
    )
    .filter(pl.col('Duration_Minutes').is_between(0, 1440))
)

preprocessing_audit = (
    pl.DataFrame({
        'Stage': [
            'Stratified sample',
            'Core fields present',
            'Valid duration (0-1440 minutes)',
        ],
        'Rows': [
            sampled_accidents.height,
            core_clean_accidents.height,
            duration_clean_accidents.height,
        ],
    })
    .with_columns(
        (pl.col('Rows').shift(1) - pl.col('Rows'))
        .fill_null(0)
        .alias('Rows_Removed_From_Previous_Stage')
    )
)

display(preprocessing_audit)

del sampled_accidents, parsed_accidents, core_clean_accidents

Stage,Rows,Rows_Removed_From_Previous_Stage
str,i64,i64
"""Stratified sample""",1545678,0
"""Core fields present""",1545643,35
"""Valid duration (0-1440 minutes…",1538632,7011


## 6. Feature Engineering

The derived fields make time, weather, and geographic comparisons easier to read.

- Weekend means Saturday or Sunday. Polars numbers Monday as 1 and Sunday as 7, so weekend is Weekday >= 6.
- Time periods are defined here as Morning (05:00-11:59), Afternoon (12:00-16:59), Evening (17:00-20:59), and Night (21:00-04:59).
- Is_High_Impact is a project-created grouping for Severity 3-4 records. It describes the dataset's traffic-impact severity level; it does not measure injuries or fatalities.
- Weather groups combine many detailed labels. Snow/Ice is checked before Rain so descriptions such as freezing rain are not misclassified as ordinary rain.

In [8]:
start_hour = pl.col('Start_Time').dt.hour()
weather_text = pl.col('Weather_Condition').fill_null('').str.to_lowercase()

accidents = duration_clean_accidents.with_columns([
    pl.col('Start_Time').dt.year().alias('Year'),
    pl.col('Start_Time').dt.month().alias('Month'),
    pl.col('Start_Time').dt.strftime('%b').alias('Month_Name'),
    start_hour.alias('Hour'),
    pl.col('Start_Time').dt.weekday().alias('Weekday'),
    pl.col('Start_Time').dt.strftime('%A').alias('Weekday_Name'),
])

accidents = accidents.with_columns([
    (pl.col('Weekday') >= 6).alias('Is_Weekend'),
    (pl.col('Severity') >= 3).alias('Is_High_Impact'),
    (
        pl.when(pl.col('Hour').is_between(5, 11)).then(pl.lit('Morning'))
        .when(pl.col('Hour').is_between(12, 16)).then(pl.lit('Afternoon'))
        .when(pl.col('Hour').is_between(17, 20)).then(pl.lit('Evening'))
        .otherwise(pl.lit('Night'))
        .alias('Time_Period')
    ),
    (
        pl.when(pl.col('Weather_Condition').is_null()).then(pl.lit('Unknown'))
        .when(weather_text.str.contains(r'snow|sleet|ice|freezing')).then(pl.lit('Snow/Ice'))
        .when(weather_text.str.contains(r'thunder|tornado|squall|storm')).then(pl.lit('Storm'))
        .when(weather_text.str.contains(r'rain|drizzle|shower')).then(pl.lit('Rain'))
        .when(weather_text.str.contains(r'fog|mist|haze|smoke')).then(pl.lit('Low Visibility'))
        .when(weather_text.str.contains(r'cloud|overcast')).then(pl.lit('Cloudy'))
        .when(weather_text.str.contains(r'clear|fair')).then(pl.lit('Clear/Fair'))
        .otherwise(pl.lit('Other'))
        .alias('Weather_Group')
    ),
    pl.concat_str(['City', pl.lit(', '), 'State']).alias('City_State'),
])

del duration_clean_accidents
display(accidents.head())

ID,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,State,Temperature(F),Humidity(%),Visibility(mi),Wind_Speed(mph),Weather_Condition,Crossing,Junction,Traffic_Signal,Sunrise_Sunset,Duration_Minutes,Year,Month,Month_Name,Hour,Weekday,Weekday_Name,Is_Weekend,Is_High_Impact,Time_Period,Weather_Group,City_State
str,i64,datetime[μs],datetime[μs],f64,f64,f64,str,str,f64,f64,f64,f64,str,bool,bool,bool,str,f64,i32,i8,str,i8,i8,str,bool,bool,str,str,str
"""A-3182822""",3,2017-11-13 13:35:57,2017-11-13 14:15:00,29.660276,-95.557602,0.0,"""Houston""","""TX""",79.0,44.0,10.0,4.6,"""Clear""",false,false,false,"""Day""",39.05,2017,11,"""Nov""",13,1,"""Monday""",false,true,"""Afternoon""","""Clear/Fair""","""Houston, TX"""
"""A-1197639""",3,2021-01-11 06:43:02,2021-01-11 07:28:21,42.401466,-71.096245,0.0,"""Medford""","""MA""",26.0,69.0,10.0,6.0,"""Mostly Cloudy""",false,false,false,"""Night""",45.316667,2021,1,"""Jan""",6,1,"""Monday""",false,true,"""Morning""","""Cloudy""","""Medford, MA"""
"""A-7666949""",3,2017-10-03 06:27:16,2017-10-03 12:27:16,33.792819,-118.087984,0.172,"""Los Alamitos""","""CA""",64.4,83.0,10.0,4.6,"""Mostly Cloudy""",false,false,false,"""Night""",360.0,2017,10,"""Oct""",6,2,"""Tuesday""",false,true,"""Morning""","""Cloudy""","""Los Alamitos, CA"""
"""A-2785572""",3,2018-06-27 09:49:22,2018-06-27 10:33:59,33.816277,-84.250694,0.0,"""Clarkston""","""GA""",78.1,76.0,10.0,null,"""Clear""",false,false,false,"""Day""",44.616667,2018,6,"""Jun""",9,3,"""Wednesday""",false,true,"""Morning""","""Clear/Fair""","""Clarkston, GA"""
"""A-3109361""",3,2017-12-06 12:49:19,2017-12-06 13:18:47,38.27203,-85.747375,0.0,"""Jeffersonville""","""IN""",45.0,46.0,10.0,12.7,"""Clear""",false,false,false,"""Day""",29.466667,2017,12,"""Dec""",12,3,"""Wednesday""",false,true,"""Afternoon""","""Clear/Fair""","""Jeffersonville, IN"""


## 7. Processed-Data Validation

These assertions are intentional checks on the assumptions used by later analyses. If an assertion fails, the notebook stops close to the problem instead of producing misleading charts or clusters.

In [9]:
assert accidents.height > 0, 'No rows remain after preprocessing.'

for required_column in CORE_REQUIRED:
    assert accidents[required_column].null_count() == 0, (
        f'Unexpected nulls in required column: {required_column}'
    )

assert set(accidents['Severity'].unique().to_list()).issubset({1, 2, 3, 4})
assert accidents['Start_Lat'].is_between(-90, 90).all()
assert accidents['Start_Lng'].is_between(-180, 180).all()
assert accidents['Duration_Minutes'].is_between(0, 1440).all()
assert accidents['Month'].is_between(1, 12).all()
assert accidents['Hour'].is_between(0, 23).all()
assert accidents['Weekday'].is_between(1, 7).all()
assert accidents['Weather_Group'].null_count() == 0
assert set(accidents['Time_Period'].unique().to_list()).issubset(
    set(TIME_PERIOD_ORDER)
)

weekend_mismatches = accidents.filter(
    pl.col('Is_Weekend') != pl.col('Weekday').is_between(6, 7)
).height
assert weekend_mismatches == 0

severity_after_cleaning = (
    accidents.group_by('Severity')
    .len()
    .rename({'len': 'Processed_Rows'})
    .sort('Severity')
)

severity_stage_audit = (
    sampling_audit
    .join(severity_after_cleaning, on='Severity')
)

display(severity_stage_audit)
print(f'All validation checks passed for {accidents.height:,} processed rows.')

Severity,Source_Rows,Sampled_Rows,Sampled_%,Processed_Rows
i64,u32,u32,f64,u32
1,67366,13473,20.0,13471
2,6156981,1231396,20.0,1225243
3,1299337,259867,20.0,259779
4,204710,40942,20.0,40139


All validation checks passed for 1,538,632 processed rows.


## 8. Parquet Checkpoint

The processed sample is saved once as Parquet for PySpark. Parquet preserves data types and is faster for repeated analytical reads than parsing the original CSV again.

The small compatibility function converts unsigned Polars integers because Spark's Parquet reader does not support every unsigned integer type.

In [10]:
def to_spark_compatible(df: pl.DataFrame) -> pl.DataFrame:
    """Cast unsigned Polars integers to Spark-compatible signed integers."""
    unsigned_types = {pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64}
    unsigned_columns = [
        pl.col(column_name).cast(pl.Int64).alias(column_name)
        for column_name, dtype in zip(df.columns, df.dtypes)
        if dtype in unsigned_types
    ]
    return df.with_columns(unsigned_columns) if unsigned_columns else df

CONFIG['artifact_dir'].mkdir(parents=True, exist_ok=True)
to_spark_compatible(accidents).write_parquet(CONFIG['processed_path'])

print(f"Processed Parquet written to: {CONFIG['processed_path']}")

Processed Parquet written to: outputs\accidents_processed.parquet


## 9. Aggregation and Analysis

The main Polars group_by operations remain visible in the notebook so each analytical table can be studied directly.

In [11]:
# Severity and location summaries
severity_counts = (
    accidents.group_by('Severity')
    .len()
    .rename({'len': 'Accident_Count'})
    .with_columns(
        (pl.col('Accident_Count') / accidents.height * 100)
        .round(2)
        .alias('Percentage')
    )
    .sort('Severity')
)

top_states = (
    accidents.group_by('State')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Accident_Count', descending=True)
    .head(15)
)

top_cities = (
    accidents.group_by('City_State')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Accident_Count', descending=True)
    .head(15)
)

display(severity_counts)
display(top_states.head())

Severity,Accident_Count,Percentage
i64,u32,f64
1,13471,0.88
2,1225243,79.63
3,259779,16.88
4,40139,2.61


State,Accident_Count
str,u32
"""CA""",347445
"""FL""",175409
"""TX""",116739
"""SC""",76072
"""NY""",69406


In [12]:
# Time summaries
accidents_by_year = (
    accidents.group_by('Year')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Year')
)

accidents_by_month = (
    accidents.group_by('Month', 'Month_Name')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Month')
)

accidents_by_hour = (
    accidents.group_by('Hour')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Hour')
)

accidents_by_weekday = (
    accidents.group_by('Weekday', 'Weekday_Name')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Weekday')
)

accidents_by_time_period = (
    accidents.group_by('Time_Period')
    .len()
    .rename({'len': 'Accident_Count'})
    .with_columns(
        pl.col('Time_Period')
        .cast(pl.Enum(TIME_PERIOD_ORDER))
        .alias('Time_Period')
    )
    .sort('Time_Period')
)

display(accidents_by_time_period)

Time_Period,Accident_Count
enum,u32
"""Morning""",570259
"""Afternoon""",459584
"""Evening""",304819
"""Night""",203970


In [13]:
# Weather summaries use available values for each relevant field.
weather_group_counts = (
    accidents.group_by('Weather_Group')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Accident_Count', descending=True)
)

day_night_counts = (
    accidents.drop_nulls(['Sunrise_Sunset'])
    .group_by('Sunrise_Sunset')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort('Accident_Count', descending=True)
)

weather_condition_audit = (
    accidents.group_by('Weather_Group', 'Weather_Condition')
    .len()
    .rename({'len': 'Accident_Count'})
    .sort(
        ['Weather_Group', 'Accident_Count'],
        descending=[False, True],
    )
    .group_by('Weather_Group', maintain_order=True)
    .head(5)
)

weather_labels = {
    'Temperature(F)': 'Temperature (°F)',
    'Humidity(%)': 'Humidity (%)',
    'Visibility(mi)': 'Visibility (mi)',
    'Wind_Speed(mph)': 'Wind speed (mph)',
}

weather_summaries = []
for weather_column in WEATHER_COLUMNS:
    summary = (
        accidents.group_by('Severity')
        .agg([
            pl.col(weather_column).count().alias('Valid_Rows'),
            pl.col(weather_column).median().round(2).alias('Median'),
            pl.col(weather_column).quantile(0.25).round(2).alias('Q1'),
            pl.col(weather_column).quantile(0.75).round(2).alias('Q3'),
        ])
        .with_columns(pl.lit(weather_labels[weather_column]).alias('Metric'))
    )
    weather_summaries.append(summary)

weather_by_severity = (
    pl.concat(weather_summaries)
    .sort(['Metric', 'Severity'])
)

In [14]:
# Duration is summarized without automatically removing statistical outliers.
duration_by_severity = (
    accidents.group_by('Severity')
    .agg([
        pl.len().alias('Accident_Count'),
        pl.col('Duration_Minutes').mean().round(1).alias('Mean_Duration'),
        pl.col('Duration_Minutes').median().round(1).alias('Median_Duration'),
        pl.col('Duration_Minutes').quantile(0.25).round(1).alias('Q1'),
        pl.col('Duration_Minutes').quantile(0.75).round(1).alias('Q3'),
    ])
    .with_columns(
        (pl.col('Q3') - pl.col('Q1')).round(1).alias('IQR')
    )
    .with_columns([
        (pl.col('Q3') - pl.col('Median_Duration')).alias('Q3_Minus_Median'),
        (pl.col('Median_Duration') - pl.col('Q1')).alias('Median_Minus_Q1'),
    ])
    .sort('Severity')
)

def road_feature_summary(df: pl.DataFrame) -> pl.DataFrame:
    """Compare present/absent outcomes consistently for each road feature."""
    summaries = []

    for feature in ROAD_FEATURES:
        feature_summary = (
            df.drop_nulls([feature])
            .with_columns(
                pl.when(pl.col(feature))
                .then(pl.lit('Present'))
                .otherwise(pl.lit('Absent'))
                .alias('Status')
            )
            .group_by('Status')
            .agg([
                pl.len().alias('Accident_Count'),
                (pl.col('Is_High_Impact').mean() * 100)
                .round(2)
                .alias('High_Impact_%'),
                pl.col('Duration_Minutes')
                .median()
                .round(1)
                .alias('Median_Duration_min'),
            ])
            .with_columns(pl.lit(feature).alias('Feature'))
        )
        summaries.append(feature_summary)

    return (
        pl.concat(summaries)
        .select(
            'Feature',
            'Status',
            'Accident_Count',
            'High_Impact_%',
            'Median_Duration_min',
        )
        .sort(['Feature', 'Status'])
    )

road_feature_results = road_feature_summary(accidents)

display(duration_by_severity)
display(road_feature_results)

Severity,Accident_Count,Mean_Duration,Median_Duration,Q1,Q3,IQR,Q3_Minus_Median,Median_Minus_Q1
i64,u32,f64,f64,f64,f64,f64,f64,f64
1,13471,45.6,44.8,29.7,59.7,30.0,14.9,15.1
2,1225243,114.6,77.4,44.4,128.8,84.4,51.4,33.0
3,259779,67.8,44.4,29.6,60.0,30.4,15.6,14.8
4,40139,185.3,126.8,60.0,360.0,300.0,233.2,66.8


Feature,Status,Accident_Count,High_Impact_%,Median_Duration_min
str,str,u32,f64,f64
"""Crossing""","""Absent""",1364071,21.1,75.0
"""Crossing""","""Present""",174561,6.96,59.8
"""Junction""","""Absent""",1425345,18.89,74.6
"""Junction""","""Present""",113287,27.06,76.7
"""Traffic_Signal""","""Absent""",1309731,21.24,75.4
"""Traffic_Signal""","""Present""",228901,9.48,54.3


## 10. Visualizations

### Severity, geography, and time

These charts compare reported frequencies in the sampled dataset. They are not adjusted risk estimates because traffic volume and other exposure measures are unavailable. The source ends in March 2023, so 2023 is a partial year and should not be compared directly with complete years.

In [15]:
def present_figure(fig, filename: str):
    """Apply one visual style, optionally export HTML, and display the figure."""
    fig.update_layout(template='plotly_white', title_x=0.02)

    if CONFIG['export_figures']:
        fig.write_html(
            CONFIG['artifact_dir'] / filename,
            include_plotlyjs='cdn',
        )

    fig.show()

severity_plot = severity_counts.with_columns(
    pl.col('Severity').cast(pl.Utf8).alias('Severity_Label')
)
fig = px.bar(
    severity_plot,
    x='Severity_Label',
    y='Accident_Count',
    color='Severity_Label',
    text='Percentage',
    title='Reported Accidents by Severity',
    labels={'Severity_Label': 'Severity'},
)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
present_figure(fig, '01_severity.html')

for chart_data, category, title, filename, color_scale in [
    (
        top_states,
        'State',
        'Top 15 States by Reported Accident Count',
        '02_states.html',
        'Viridis',
    ),
    (
        top_cities,
        'City_State',
        'Top 15 Cities by Reported Accident Count',
        '03_cities.html',
        'Blues',
    ),
]:
    fig = px.bar(
        chart_data.sort('Accident_Count'),
        x='Accident_Count',
        y=category,
        orientation='h',
        color='Accident_Count',
        color_continuous_scale=color_scale,
        title=title,
        labels={'Accident_Count': 'Accident count'},
    )
    present_figure(fig, filename)

In [16]:
fig = px.line(
    accidents_by_year,
    x='Year',
    y='Accident_Count',
    markers=True,
    title='Reported Accidents by Year (2023 is Partial)',
)
present_figure(fig, '04_year.html')

fig = px.bar(
    accidents_by_month,
    x='Month_Name',
    y='Accident_Count',
    color='Accident_Count',
    title='Reported Accidents by Month',
    category_orders={
        'Month_Name': accidents_by_month['Month_Name'].to_list(),
    },
)
present_figure(fig, '05_month.html')

fig = px.line(
    accidents_by_hour,
    x='Hour',
    y='Accident_Count',
    markers=True,
    title='Reported Accidents by Hour',
)
fig.add_vrect(
    x0=7,
    x1=9,
    fillcolor='orange',
    opacity=0.1,
    annotation_text='AM rush',
)
fig.add_vrect(
    x0=16,
    x1=18,
    fillcolor='red',
    opacity=0.1,
    annotation_text='PM rush',
)
present_figure(fig, '06_hour.html')

fig = px.bar(
    accidents_by_weekday,
    x='Weekday_Name',
    y='Accident_Count',
    color='Accident_Count',
    title='Reported Accidents by Day of Week',
    category_orders={
        'Weekday_Name': accidents_by_weekday['Weekday_Name'].to_list(),
    },
)
present_figure(fig, '07_weekday.html')

fig = px.bar(
    accidents_by_time_period,
    x='Time_Period',
    y='Accident_Count',
    color='Accident_Count',
    title='Reported Accidents by Time Period',
    category_orders={'Time_Period': TIME_PERIOD_ORDER},
)
present_figure(fig, '08_time_period.html')

### Weather

Weather accident counts show how frequently conditions appear among reported accidents; they do not estimate weather-related risk. The dataset does not provide the total number of hours, days, trips, or miles driven under each weather condition. Numeric summaries therefore report their valid row counts and do not remove an otherwise useful accident when one weather measurement is missing.

In [17]:
fig = px.bar(
    weather_group_counts,
    x='Weather_Group',
    y='Accident_Count',
    color='Accident_Count',
    title='Reported Accidents by Weather Group',
)
present_figure(fig, '09_weather_groups.html')

print('Five most common original conditions inside each weather group')
display(weather_condition_audit)

fig = px.bar(
    day_night_counts,
    x='Sunrise_Sunset',
    y='Accident_Count',
    color='Sunrise_Sunset',
    title='Reported Accidents During Day and Night',
)
present_figure(fig, '10_day_night.html')

weather_plot = weather_by_severity.with_columns(
    pl.col('Severity').cast(pl.Utf8).alias('Severity_Label')
)
display(weather_plot)

fig = px.bar(
    weather_plot,
    x='Severity_Label',
    y='Median',
    color='Severity_Label',
    facet_col='Metric',
    facet_col_wrap=2,
    hover_data=['Valid_Rows', 'Q1', 'Q3'],
    title='Median Recorded Weather Measurements by Severity',
    labels={'Severity_Label': 'Severity'},
)
fig.update_yaxes(matches=None, showticklabels=True)
present_figure(fig, '11_numeric_weather.html')

Five most common original conditions inside each weather group


Weather_Group,Weather_Condition,Accident_Count
str,str,u32
"""Clear/Fair""","""Fair""",508713
"""Clear/Fair""","""Clear""",161753
"""Clear/Fair""","""Fair / Windy""",7139
"""Cloudy""","""Mostly Cloudy""",202188
"""Cloudy""","""Cloudy""",162398
"""Cloudy""","""Partly Cloudy""",139402
"""Cloudy""","""Overcast""",76293
"""Cloudy""","""Scattered Clouds""",40934
"""Low Visibility""","""Fog""",19628


Severity,Valid_Rows,Median,Q1,Q3,Metric,Severity_Label
i64,u32,f64,f64,f64,str,str
1,13212,66.0,42.0,84.0,"""Humidity (%)""","""1"""
2,1197452,67.0,48.0,84.0,"""Humidity (%)""","""2"""
3,254283,67.0,49.0,84.0,"""Humidity (%)""","""3"""
4,38762,70.0,51.0,87.0,"""Humidity (%)""","""4"""
1,13235,73.0,64.0,82.0,"""Temperature (°F)""","""1"""
2,1199098,64.0,49.0,76.0,"""Temperature (°F)""","""2"""
3,254558,64.9,50.0,76.0,"""Temperature (°F)""","""3"""
4,38828,60.1,43.0,73.9,"""Temperature (°F)""","""4"""
1,13287,10.0,10.0,10.0,"""Visibility (mi)""","""1"""


### Duration and road features

Mean, median, Q1, Q3, and IQR are descriptive statistics only. They are not used as an automatic outlier-removal rule.

Road-feature results compare records where a feature is marked present with records where it is marked absent. These are observational associations, not causal effects. Differences may reflect location, road type, reporting practices, traffic, or other unmeasured factors. Severity 3-4 is the project's higher-impact grouping, not an injury or fatality measure.

In [18]:
fig = px.bar(
    duration_by_severity,
    x='Severity',
    y='Median_Duration',
    color='Median_Duration',
    error_y='Q3_Minus_Median',
    error_y_minus='Median_Minus_Q1',
    hover_data=['Accident_Count', 'Mean_Duration', 'Q1', 'Q3', 'IQR'],
    title='Median Recorded Accident Duration by Severity (Q1-Q3)',
    labels={'Median_Duration': 'Median duration (minutes)'},
)
present_figure(fig, '12_duration.html')

fig = px.bar(
    road_feature_results,
    x='Feature',
    y='High_Impact_%',
    color='Status',
    barmode='group',
    text='High_Impact_%',
    hover_data=['Accident_Count', 'Median_Duration_min'],
    title='Higher-Impact Record Share by Road Feature',
    labels={'High_Impact_%': 'Severity 3-4 records (%)'},
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
present_figure(fig, '13_road_features.html')

### Geographic distribution

The map uses a fixed-size reproducible sample for responsiveness. Dense-looking areas reflect reported counts, not risk, because traffic and travel exposure are unavailable.

In [19]:
map_size = min(CONFIG['map_sample_size'], accidents.height)
map_sample = (
    accidents.sample(n=map_size, seed=CONFIG['seed'])
    .with_columns(
        pl.col('Severity').cast(pl.Utf8).alias('Severity_Label')
    )
)

fig = px.scatter_map(
    map_sample,
    lat='Start_Lat',
    lon='Start_Lng',
    color='Severity_Label',
    hover_data=[
        'City',
        'State',
        'Weather_Group',
        'Duration_Minutes',
    ],
    zoom=3,
    height=650,
    map_style='open-street-map',
    title=f'Accident Locations ({map_size:,}-Row Reproducible Sample)',
)
fig.update_layout(margin={'r': 0, 't': 55, 'l': 0, 'b': 0})
present_figure(fig, '14_accident_map.html')

## 11. PySpark K-Means Geographic Clustering

This is the notebook's only K-Means implementation.

Latitude and longitude remain unscaled so both coordinate dimensions stay in the same degree-based units. This is a simple, explainable approximation: Euclidean distance on latitude/longitude is not spherical distance and does not represent road-network distance.

The notebook tests K = 3 through K = 12 and selects the model with the highest Silhouette Score. The output clusters are broad geographic groupings called Regional_Cluster, not true accident hotspots. A cluster with a higher average recorded Severity is not automatically a more dangerous region.

In [20]:
# Read the Parquet checkpoint with Spark and assemble the two coordinates.
spark = (
    SparkSession.builder
    .appName('USAccidentsRegionalClustering')
    .master('local[*]')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

accidents_spark = (
    spark.read.parquet(str(CONFIG['processed_path']))
    .select(
        'ID',
        'Start_Lat',
        'Start_Lng',
        'State',
        'City',
        'Severity',
        'Weather_Group',
        'Duration_Minutes',
        col('Distance(mi)'),
    )
    .dropna(subset=['Start_Lat', 'Start_Lng'])
)

coordinate_assembler = VectorAssembler(
    inputCols=['Start_Lat', 'Start_Lng'],
    outputCol='features',
)

coordinate_features = coordinate_assembler.transform(accidents_spark).cache()
spark_row_count = coordinate_features.count()

print(f'Rows available to Spark K-Means: {spark_row_count:,}')

Rows available to Spark K-Means: 1,538,632


In [21]:
# Fit one model for each candidate K and evaluate it on the same data.
cluster_evaluator = ClusteringEvaluator(
    featuresCol='features',
    predictionCol='Regional_Cluster',
)

k_models = {}
k_score_rows = []

for k in CONFIG['k_values']:
    model = KMeans(
        featuresCol='features',
        predictionCol='Regional_Cluster',
        k=k,
        seed=CONFIG['seed'],
    ).fit(coordinate_features)

    predictions = model.transform(coordinate_features)
    silhouette_score = cluster_evaluator.evaluate(predictions)

    k_models[k] = model
    k_score_rows.append({
        'K': k,
        'Silhouette': silhouette_score,
    })
    print(f'K={k}: silhouette={silhouette_score:.4f}')

silhouette_scores = pl.DataFrame(k_score_rows).sort('K')
best_score = max(k_score_rows, key=lambda row: row['Silhouette'])
best_k = int(best_score['K'])
best_model = k_models[best_k]

print(f'Selected K={best_k} with silhouette={best_score["Silhouette"]:.4f}')

K=3: silhouette=0.6297
K=4: silhouette=0.6614
K=5: silhouette=0.6383
K=6: silhouette=0.6613
K=7: silhouette=0.6718
K=8: silhouette=0.6886
K=9: silhouette=0.7081
K=10: silhouette=0.7430
K=11: silhouette=0.7263
K=12: silhouette=0.6468
Selected K=10 with silhouette=0.7430


In [22]:
# Apply the selected model once and build a readable regional profile.
clustered_spark = best_model.transform(coordinate_features).cache()
clustered_spark.count()

cluster_profile_spark = clustered_spark.groupBy('Regional_Cluster').agg(
    count('*').alias('Accident_Count'),
    avg('Severity').alias('Avg_Severity'),
    avg('Duration_Minutes').alias('Avg_Duration_min'),
    avg('Distance(mi)').alias('Avg_Distance_mi'),
)

# There are only K x number-of-states rows here, so collecting them is small.
state_rows = (
    clustered_spark.groupBy('Regional_Cluster', 'State')
    .count()
    .orderBy('Regional_Cluster', col('count').desc())
    .collect()
)

top_states_by_cluster = {}
for row in state_rows:
    cluster_id = int(row['Regional_Cluster'])
    top_states_by_cluster.setdefault(cluster_id, [])

    if len(top_states_by_cluster[cluster_id]) < 3:
        top_states_by_cluster[cluster_id].append(row['State'])

cluster_centers = best_model.clusterCenters()
cluster_profile_rows = []

for row in cluster_profile_spark.collect():
    values = row.asDict()
    cluster_id = int(values['Regional_Cluster'])
    values['Center_Lat'] = float(cluster_centers[cluster_id][0])
    values['Center_Lng'] = float(cluster_centers[cluster_id][1])
    values['Top_States'] = ', '.join(
        top_states_by_cluster.get(cluster_id, [])
    )
    cluster_profile_rows.append(values)

cluster_profile = (
    pl.DataFrame(cluster_profile_rows)
    .sort('Accident_Count', descending=True)
)

In [23]:
# Keep compact cluster counts for dashboard filters.
cluster_filter_rows = (
    clustered_spark.groupBy(
        'State',
        'Severity',
        'Weather_Group',
        'Regional_Cluster',
    )
    .count()
    .collect()
)

cluster_filter_counts = (
    pl.DataFrame([row.asDict() for row in cluster_filter_rows])
    .rename({'count': 'Accident_Count'})
)

# Collect only a bounded point sample for maps and dashboard interaction.
cluster_map_target = min(
    CONFIG['cluster_map_sample_size'],
    spark_row_count,
)
cluster_map_fraction = min(
    1.0,
    cluster_map_target * 1.5 / spark_row_count,
)

cluster_map_rows = (
    clustered_spark.sample(
        withReplacement=False,
        fraction=cluster_map_fraction,
        seed=CONFIG['seed'],
    )
    .select(
        'Start_Lat',
        'Start_Lng',
        'State',
        'City',
        'Severity',
        'Weather_Group',
        'Duration_Minutes',
        'Regional_Cluster',
    )
    .limit(cluster_map_target)
    .collect()
)

cluster_map_sample = pl.DataFrame([
    row.asDict() for row in cluster_map_rows
])

# Release cached Spark data and close the local session.
clustered_spark.unpersist()
coordinate_features.unpersist()
spark.stop()

print('Spark clustering complete and session stopped.')

Spark clustering complete and session stopped.


In [24]:
print(f'Selected number of regional clusters: K={best_k}')
display(
    silhouette_scores.with_columns(
        pl.col('Silhouette').round(4)
    )
)
display(
    cluster_profile.with_columns([
        pl.col('Avg_Severity').round(3),
        pl.col('Avg_Duration_min').round(1),
        pl.col('Avg_Distance_mi').round(3),
        pl.col('Center_Lat').round(3),
        pl.col('Center_Lng').round(3),
    ])
)

fig = px.line(
    silhouette_scores,
    x='K',
    y='Silhouette',
    markers=True,
    title='K-Means Model Selection by Silhouette Score',
)
present_figure(fig, '15_silhouette.html')

fig = px.bar(
    cluster_profile,
    x='Regional_Cluster',
    y='Accident_Count',
    color='Avg_Severity',
    hover_data=['Top_States', 'Center_Lat', 'Center_Lng'],
    title='Broad Geographic Regions: Count and Average Severity',
)
present_figure(fig, '16_regions.html')

cluster_map_plot = cluster_map_sample.with_columns(
    pl.col('Regional_Cluster').cast(pl.Utf8).alias('Region_Label')
)
fig = px.scatter_map(
    cluster_map_plot,
    lat='Start_Lat',
    lon='Start_Lng',
    color='Region_Label',
    hover_data=[
        'City',
        'State',
        'Severity',
        'Weather_Group',
        'Duration_Minutes',
    ],
    zoom=3,
    height=650,
    map_style='open-street-map',
    title='PySpark K-Means Broad Geographic Regions (Sample)',
)
fig.update_layout(margin={'r': 0, 't': 55, 'l': 0, 'b': 0})
present_figure(fig, '17_region_map.html')

Selected number of regional clusters: K=10


K,Silhouette
i64,f64
3,0.6297
4,0.6614
5,0.6383
6,0.6613
7,0.6718
8,0.6886
9,0.7081
10,0.743
11,0.7263


Regional_Cluster,Accident_Count,Avg_Severity,Avg_Duration_min,Avg_Distance_mi,Center_Lat,Center_Lng,Top_States
i64,i64,f64,f64,f64,f64,f64,str
2,346754,2.168,110.7,0.484,35.595,-119.443,"""CA, NV, AZ"""
0,278053,2.259,108.1,0.741,40.256,-75.491,"""NY, PA, VA"""
1,187255,2.199,97.0,0.397,34.777,-81.187,"""SC, NC, GA"""
5,168108,2.143,130.1,0.533,27.4,-81.255,"""FL, GA"""
8,136788,2.208,91.0,0.282,31.644,-96.538,"""TX, OK, LA"""
6,121262,2.383,97.2,0.509,41.274,-85.794,"""IL, MI, OH"""
9,89339,2.198,95.7,0.543,33.129,-88.052,"""TN, LA, AL"""
7,80399,2.207,98.4,0.802,36.94,-109.841,"""AZ, UT, CO"""
3,66004,2.177,156.2,0.819,45.353,-121.626,"""OR, WA, CA"""


## 12. Interactive Dash Dashboard

The dashboard is kept as a function because a Dash layout and callback form one reusable application component. Its filters apply to the KPIs, charts, road-feature comparison, regional counts, and sampled maps. K-Means is never refitted inside a callback.

The server is disabled by default so Restart Kernel -> Run All can finish without leaving a dashboard process running. Set RUN_DASHBOARD = True and rerun the launch cell when you want to use it.

In [25]:
def filter_frame(
    df: pl.DataFrame,
    state,
    severity,
    weather,
) -> pl.DataFrame:
    """Apply the three dashboard filters to a Polars DataFrame."""
    if state != 'All':
        df = df.filter(pl.col('State') == state)
    if severity != 'All':
        df = df.filter(pl.col('Severity') == severity)
    if weather != 'All':
        df = df.filter(pl.col('Weather_Group') == weather)
    return df

def mean_label(series: pl.Series, decimals: int) -> str:
    """Format a nullable mean for a dashboard KPI."""
    value = series.mean()
    return 'N/A' if value is None else f'{value:.{decimals}f}'

def build_dashboard(
    df: pl.DataFrame,
    regional_counts: pl.DataFrame,
    regional_map_sample: pl.DataFrame,
) -> Dash:
    app = Dash(__name__)

    filter_options = {
        'state': ['All'] + sorted(df['State'].unique().to_list()),
        'severity': ['All'] + sorted(df['Severity'].unique().to_list()),
        'weather': ['All'] + sorted(df['Weather_Group'].unique().to_list()),
    }

    app.layout = html.Div(
        style={'fontFamily': 'Arial', 'padding': '20px'},
        children=[
            html.H1(
                'US Accidents Analytics Dashboard',
                style={'textAlign': 'center'},
            ),
            html.P(
                'Reported counts; regional maps use bounded samples.',
                style={'textAlign': 'center'},
            ),
            html.Div(
                style={'display': 'flex', 'gap': '16px'},
                children=[
                    html.Div(
                        [
                            html.Label('State'),
                            dcc.Dropdown(
                                filter_options['state'],
                                'All',
                                id='state',
                            ),
                        ],
                        style={'flex': 1},
                    ),
                    html.Div(
                        [
                            html.Label('Severity'),
                            dcc.Dropdown(
                                filter_options['severity'],
                                'All',
                                id='severity',
                            ),
                        ],
                        style={'flex': 1},
                    ),
                    html.Div(
                        [
                            html.Label('Weather'),
                            dcc.Dropdown(
                                filter_options['weather'],
                                'All',
                                id='weather',
                            ),
                        ],
                        style={'flex': 1},
                    ),
                ],
            ),
            html.Div(
                id='kpis',
                style={'display': 'flex', 'gap': '12px'},
            ),
            dcc.Tabs(
                id='tab',
                value='overview',
                children=[
                    dcc.Tab(label='Overview', value='overview'),
                    dcc.Tab(label='Time & Weather', value='time'),
                    dcc.Tab(label='Road & Regions', value='regions'),
                    dcc.Tab(label='Map', value='map'),
                ],
            ),
            html.Div(id='tab-content'),
        ],
    )

    def kpi_card(title, value, color):
        return html.Div(
            style={
                'flex': 1,
                'padding': '14px',
                'borderRadius': '8px',
                'backgroundColor': color,
                'color': 'white',
                'textAlign': 'center',
            },
            children=[html.H2(value), html.P(title)],
        )

    @app.callback(
        Output('kpis', 'children'),
        Output('tab-content', 'children'),
        Input('state', 'value'),
        Input('severity', 'value'),
        Input('weather', 'value'),
        Input('tab', 'value'),
    )
    def update_dashboard(state, severity, weather, selected_tab):
        current = filter_frame(df, state, severity, weather)

        if current.is_empty():
            return [], html.H3('No records match the selected filters.')

        cards = [
            kpi_card(
                'Reported accidents',
                f'{current.height:,}',
                '#2980b9',
            ),
            kpi_card(
                'Average severity',
                mean_label(current['Severity'], 2),
                '#c0392b',
            ),
            kpi_card(
                'Median duration',
                f"{current['Duration_Minutes'].median():.1f}",
                '#27ae60',
            ),
            kpi_card(
                'Average distance',
                mean_label(current['Distance(mi)'], 2),
                '#8e44ad',
            ),
        ]

        if selected_tab == 'overview':
            figures = [
                px.bar(
                    current.group_by('Severity').len().sort('Severity'),
                    x='Severity',
                    y='len',
                    title='Severity',
                ),
                px.bar(
                    current.group_by('State')
                    .len()
                    .sort('len', descending=True)
                    .head(10)
                    .sort('len'),
                    x='len',
                    y='State',
                    orientation='h',
                    title='Top states',
                ),
                px.bar(
                    current.group_by('City_State')
                    .len()
                    .sort('len', descending=True)
                    .head(10)
                    .sort('len'),
                    x='len',
                    y='City_State',
                    orientation='h',
                    title='Top cities',
                ),
            ]

        elif selected_tab == 'time':
            figures = [
                px.line(
                    current.group_by('Hour').len().sort('Hour'),
                    x='Hour',
                    y='len',
                    markers=True,
                    title='By hour',
                ),
                px.bar(
                    current.group_by('Weather_Group').len(),
                    x='Weather_Group',
                    y='len',
                    title='Weather',
                ),
                px.bar(
                    current.drop_nulls(['Sunrise_Sunset'])
                    .group_by('Sunrise_Sunset')
                    .len(),
                    x='Sunrise_Sunset',
                    y='len',
                    title='Day and night',
                ),
            ]

        elif selected_tab == 'regions':
            road_figure = px.bar(
                road_feature_summary(current),
                x='Feature',
                y='High_Impact_%',
                color='Status',
                barmode='group',
                title='Road features',
            )

            filtered_regional_counts = filter_frame(
                regional_counts,
                state,
                severity,
                weather,
            )
            regional_summary = (
                filtered_regional_counts
                .with_columns(
                    (
                        pl.col('Severity') * pl.col('Accident_Count')
                    ).alias('Weighted_Severity')
                )
                .group_by('Regional_Cluster')
                .agg([
                    pl.col('Accident_Count').sum(),
                    pl.col('Weighted_Severity').sum(),
                ])
                .with_columns(
                    (
                        pl.col('Weighted_Severity')
                        / pl.col('Accident_Count')
                    ).alias('Avg_Severity')
                )
            )
            regional_figure = px.bar(
                regional_summary,
                x='Regional_Cluster',
                y='Accident_Count',
                color='Avg_Severity',
                title='Filtered broad regions',
            )

            regional_points = filter_frame(
                regional_map_sample,
                state,
                severity,
                weather,
            )
            if regional_points.is_empty():
                regional_map = px.scatter(
                    title='No sampled map points match these filters'
                )
            else:
                regional_points = regional_points.with_columns(
                    pl.col('Regional_Cluster')
                    .cast(pl.Utf8)
                    .alias('Region_Label')
                )
                regional_map = px.scatter_map(
                    regional_points,
                    lat='Start_Lat',
                    lon='Start_Lng',
                    color='Region_Label',
                    map_style='open-street-map',
                    zoom=3,
                    title='Filtered regional sample',
                )

            figures = [
                road_figure,
                regional_figure,
                regional_map,
            ]

        else:
            point_count = min(10_000, current.height)
            map_points = (
                current.sample(n=point_count, seed=CONFIG['seed'])
                .with_columns(
                    pl.col('Severity')
                    .cast(pl.Utf8)
                    .alias('Severity_Label')
                )
            )
            figures = [
                px.scatter_map(
                    map_points,
                    lat='Start_Lat',
                    lon='Start_Lng',
                    color='Severity_Label',
                    map_style='open-street-map',
                    zoom=3,
                    title=f'Filtered map ({point_count:,} rows)',
                )
            ]

        for figure in figures:
            figure.update_layout(template='plotly_white')

        graphs = [dcc.Graph(figure=figure) for figure in figures]
        return cards, html.Div(graphs)

    return app

In [31]:
dashboard = build_dashboard(
    accidents,
    cluster_filter_counts,
    cluster_map_sample,
)

RUN_DASHBOARD = False  # Set to True to launch the dashboard

if RUN_DASHBOARD:
    dashboard.run(
        debug=False,
        jupyter_mode='external',
        port=8051,
    )
else:
    print(
        'Dashboard ready. Set RUN_DASHBOARD = True and rerun '
        'this cell to launch it.'
    )

Dashboard ready. Set RUN_DASHBOARD = True and rerun this cell to launch it.


## 13. Findings and Limitations

The next cell prints findings from the current run instead of hard-coding values.

Interpret every result within these limits:

- Accident counts are frequencies, not risk estimates, because traffic volume and travel exposure are unavailable.
- Weather counts are not weather-related risk; the amount of exposure to each weather condition is unknown.
- Road-feature comparisons are observational associations and do not demonstrate causal effects.
- Regional_Cluster values are arbitrary labels for broad coordinate groupings, not true accident hotspots.
- A cluster with the highest average recorded Severity should not be called the most dangerous region.
- Severity is the dataset's traffic-impact measure; this notebook does not infer injuries or fatalities from it.
- The 20% sample is reproducible but still introduces sampling variation.
- Recorded duration may differ from actual road-clearance time, and the 24-hour validity limit is a project assumption.
- Geographic K-Means uses degree-based Euclidean distance rather than spherical or road-network distance.
- Reporting coverage varies across place and time, and 2023 contains only January through March.

In [33]:
top_state = top_states.row(0, named=True)
top_city = top_cities.row(0, named=True)
peak_hour = (
    accidents_by_hour
    .sort('Accident_Count', descending=True)
    .row(0, named=True)
)
common_weather = weather_group_counts.row(0, named=True)
highest_average_severity_cluster = (
    cluster_profile
    .sort('Avg_Severity', descending=True)
    .row(0, named=True)
)

print('Run-specific findings')
print(f"- Processed rows: {accidents.height:,}")
print(
    f"- Most reported records are in {top_state['State']}: "
    f"{top_state['Accident_Count']:,}"
)
print(
    f"- Leading city: {top_city['City_State']} "
    f"({top_city['Accident_Count']:,})"
)
print(
    f"- Peak hour: {peak_hour['Hour']}:00 "
    f"({peak_hour['Accident_Count']:,})"
)
print(
    f"- Most common weather group: "
    f"{common_weather['Weather_Group']}"
)
print(
    f'- Silhouette-selected regional cluster count: K={best_k} '
    f'(silhouette={best_score["Silhouette"]:.4f})'
)
print(
    '- Highest recorded average severity is in Regional_Cluster '
    f"{highest_average_severity_cluster['Regional_Cluster']} "
    '(descriptive only; this does not establish danger or risk).'
)
print('- All processed-data validation assertions passed.')

Run-specific findings
- Processed rows: 1,538,632
- Most reported records are in CA: 347,445
- Leading city: Miami, FL (36,955)
- Peak hour: 7:00 (117,132)
- Most common weather group: Clear/Fair
- Silhouette-selected regional cluster count: K=10 (silhouette=0.7430)
- Highest recorded average severity is in Regional_Cluster 6 (descriptive only; this does not establish danger or risk).
- All processed-data validation assertions passed.


# To open the dashboard again without rerun the whole cells

Cell 1 — save the dashboard data once

In [28]:
# Save dashboard data for quick reopening later

cluster_filter_counts.write_parquet(
    "outputs/cluster_filter_counts.parquet"
)

cluster_map_sample.write_parquet(
    "outputs/cluster_map_sample.parquet"
)

print("Dashboard data saved.")

Dashboard data saved.


Cell 2 — Quick Dashboard Launcher

In [29]:
# QUICK DASHBOARD LAUNCH
# Use this after reopening the notebook instead of rerunning the full analysis.

import polars as pl
import plotly.express as px
from dash import Dash, Input, Output, dcc, html


# Load already-prepared dashboard data
accidents = pl.read_parquet(
    "outputs/accidents_processed.parquet"
)

cluster_filter_counts = pl.read_parquet(
    "outputs/cluster_filter_counts.parquet"
)

cluster_map_sample = pl.read_parquet(
    "outputs/cluster_map_sample.parquet"
)


# Build dashboard
dashboard = build_dashboard(
    accidents,
    cluster_filter_counts,
    cluster_map_sample,
)


# Launch dashboard
dashboard.run(
    debug=False,
    jupyter_mode="external",
    port=8051,
)

Dash app running on http://127.0.0.1:8051/
